# pMF Any-to-Any · RGB e boxes · Colab T4

Seleciona um runtime T4. Esta experiência usa um diretório novo e os pesos originais pMF-L/32-512. O preflight testa a paridade inicial RGB e o JVP, faz overfit em quatro exemplos de treino e testa as quatro tarefas. O treino completo fica bloqueado até o preflight passar e começa na época 0. Os resultados ficam em /content enquanto durar a sessão.

In [ ]:
import json
from pathlib import Path
import subprocess
import sys

REPO = Path("/content/Synthetic-Data-Generation-Framework")
DATASET = Path("/content/rf100-bi3q3")
RUN = Path("/content/pmf_rgb_boxes_run")
CONFIG = "configs/pmf_rgb_boxes.json"
REPO_URL = "https://github.com/andreferraria453/Synthetic-Data-Generation-Framework.git"

subprocess.run(["nvidia-smi", "-L"], check=True)
if REPO.exists():
    subprocess.run(["git", "pull", "--ff-only", "origin", "main"], cwd=REPO, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

def run(*args, on_epoch=None):
    from collections import deque

    tail = deque(maxlen=24)
    command = [sys.executable, "-u", *map(str, args)]
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors="replace") as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                tail.append(line)
                if on_epoch is not None and line.startswith('{"epoch":'):
                    try:
                        on_epoch(json.loads(line)["epoch"])
                    except Exception as exc:
                        print(f"Falha ao atualizar o painel: {exc}", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
        status = process.wait()
    if status:
        raise RuntimeError("".join(tail))

run("-m", "pip", "install", "-r", "requirements-any-to-any.txt")
if not (DATASET / "data.yaml").is_file():
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="LibreYOLO/corrosion-bi3q3", repo_type="dataset",
        revision="d20b5b1e6b21ff7de555bbf848dcf7d30ccf8ee7", local_dir=str(DATASET),
    )

ARGS = ["-m", "rae_experiment.any_to_any.train", "--config", CONFIG,
        "--dataset", DATASET, "--output", RUN]
run("-m", "rae_experiment.any_to_any.data", "--dataset", DATASET, "--audit-only")
run("-m", "unittest", "discover", "-s", "tests", "-v")

## Auditoria e overfit

O JSON de preflight inclui o erro da comparação com o sampler original, o erro relativo do JVP, os resultados de overfit por tarefa e o pico de VRAM. Requer que as quatro losses fixas desçam. O checkpoint final começa dos pesos originais novamente, num processo novo.

In [ ]:
run(*ARGS, "--smoke")

## Curvas e exemplos por época

Define START_TRAINING como True só depois do preflight. Durante o treino, a saída mostra a época e os batches. A cada época concluída, o painel atualiza as curvas e mostra as quatro tarefas para um exemplo fixo de validação. Há dois exemplos por tarefa guardados em RUN/examples. Verde indica caixas reais e rosa indica caixas previstas.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image

handles = {}

def refresh(epoch):
    rows = [json.loads(path.read_text()) for path in sorted((RUN / "epochs").glob("*.json"))
            if int(path.stem) <= epoch]
    groups = [
        ("Loss (log)", "train/validation",
         ("loss_global", "loss_rgb", "loss_boxes", "loss_rgb_unweighted", "loss_boxes_unweighted")),
        ("Gradientes e LR", "train", ("gradient_norm", "gradient_norm_max", "learning_rate")),
        ("RGB", "metrics", ("fid", "kid")),
        ("Boxes", "metrics", ("map50", "map50_95")),
        ("Frequência das tarefas", "train", (
            "task_noise_to_rgb_boxes_frequency", "task_rgb_to_boxes_frequency",
            "task_boxes_to_rgb_frequency", "task_noise_to_rgb_frequency",
        )),
        ("Inferência (imagens/s)", "metrics", (
            "noise_to_rgb_boxes_images_per_second", "rgb_to_boxes_images_per_second",
            "boxes_to_rgb_images_per_second", "noise_to_rgb_images_per_second",
        )),
    ]
    figure, axes = plt.subplots(2, 3, figsize=(18, 8), constrained_layout=True)
    for axis, (title, splits, keys) in zip(axes.flat, groups):
        for split in splits.split("/"):
            for key in keys:
                points = [(row["epoch"], row[split].get(key)) for row in rows]
                points = [(x, y) for x, y in points if y is not None and
                          (title != "Loss (log)" or y > 0)]
                if points:
                    axis.plot(*zip(*points), marker=".", label=f"{split}/{key}")
        if title in ("Loss (log)", "Gradientes e LR"):
            axis.set_yscale("log")
        axis.set(title=title, xlabel="Época")
        if axis.lines:
            axis.legend(fontsize=7)
    figure.suptitle(f"pMF RGB + boxes · última época: {epoch} | treino: {rows[-1]['train_seconds']:.1f} s")
    directory = RUN / "examples" / f"{epoch:03}"
    figure.savefig(directory / "curvas.png", dpi=110)
    if "metrics" in handles:
        handles["metrics"].update(figure)
    else:
        handles["metrics"] = display(figure, display_id=True)
    plt.close(figure)
    paths = sorted(directory.glob("00_*.png"))
    if paths:
        with Image.open(paths[0]) as image:
            width, height = image.size
        grid = Image.new("RGB", (width * 2, height * 2))
        for index, path in enumerate(paths):
            with Image.open(path) as image:
                grid.paste(image, (index % 2 * width, index // 2 * height))
        if "samples" in handles:
            handles["samples"].update(grid)
        else:
            handles["samples"] = display(grid, display_id=True)

START_TRAINING = False
RESUME = None  # Exemplo: RUN / "checkpoints" / "epoch-005.pt"
if START_TRAINING:
    resume_args = ["--resume", RESUME] if RESUME is not None else []
    run(*ARGS, "--train", *resume_args, on_epoch=refresh)
else:
    available = sorted((RUN / "epochs").glob("*.json"))
    if available:
        refresh(int(available[-1].stem))
    print("Treino em pausa. Depois do preflight, define START_TRAINING = True.")